# Day 21 · 自动评测流水线

**配套讲义**: [`days/day-21.md`](../days/day-21.md) ｜ **需要 GPU（云机器）**

一条命令跑完全套评测并出 markdown 报告；把 LLM-as-judge 的**位置偏见**校准掉，让 judge 与人工打分的相关性 ≥ 0.7。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w4.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 先跑不花钱的规则打分

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.eval.run_eval",
                    "--model", "Qwen/Qwen2.5-VL-3B-Instruct",
                    "--eval", "data/eval/cx_eval_v1.jsonl",
                    "--tag", "base", "--no-judge", "--limit", "20"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-2500:] or r.stderr[-2500:])

## 2. judge 偏见自检（几个手造样本，不用跑模型）

In [ ]:
import sys; sys.path.insert(0, "..")
from src.eval.judge import JUDGE_SYSTEM, make_calibration_sheet

print(JUDGE_SYSTEM[:900])
print("\n" + "=" * 70)
try:
    sheet = make_calibration_sheet(n=30)
    print(f"校准表已生成 {len(sheet)} 条 → reports/judge_sheet.jsonl")
    print("→ 打开它，逐条人工打分（1–5），再跑 calibrate")
except Exception as e:
    print("生成校准表：", e)

## 3. 位置偏见的量化

造一对「实际上 A 更好」的答案，分别按 A/B 和 B/A 各打一次。
如果两次结论相反 —— 恭喜，你亲手测出了位置偏见。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.eval.judge import pairwise_with_calibration

answer_short = "M/L 码有现货，S 码预计 3 天补货。"
answer_long  = ("您好呀！非常感谢您的咨询～关于您问的这件商品呢，"
                "我想先跟您说明一下我们的库存情况哦，目前呢……"
                "（此处省略 400 字客套话）")
print("短答案明显更好（信息密度高），但 judge 会怎么选？")
try:
    print(pairwise_with_calibration(answer_short, answer_long))
except Exception as e:
    print("需要配置 JUDGE_API_KEY：", e)

## 验收清单

- [ ] 一条命令跑完全套评测，输出 markdown 报告
- [ ] 报告**按难度层和意图分组**，能看出弱在哪一层（不能只有总分）
- [ ] judge 与人工打分相关性 ≥ 0.7（做 30 条人工标注校准）
- [ ] 位置偏见已通过 A/B 交换抵消，且你能说出抵消的原理

**卡住了？** 回看 [`days/day-21.md`](../days/day-21.md) 第五节「容易踩的坑」。

> **明天**：`days/day-22.md` —— 幻觉评测：让模型学会说「不确定」